# Spark Basics, Compared with pandas

You already know **pandas**. Spark DataFrames look very similar, but they work differently: the data is split across the machines of a cluster, and nothing is computed until you ask for a result.

In this notebook every operation is shown twice: first in **pandas**, then in **Spark**. Run the cells from top to bottom (**Shift + Enter** runs a cell and moves to the next one).

We use the retail data of the course: customers (`customers.csv`) and their purchases (`transactions.csv`).

## 0. Setup

In Databricks the variable `spark` (the connection to Spark) already exists. The cell below also makes the notebook work outside Databricks (for example locally or on Colab).

In [ ]:
import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.getOrCreate()

# display(df) shows an interactive table only on Databricks: elsewhere we use show() instead
if "DATABRICKS_RUNTIME_VERSION" not in os.environ:
    def display(df):
        df.show(20, truncate=False)

Check that Spark works: this should print the numbers from 0 to 4.

In [ ]:
spark.range(5).show()

## 1. Loading data

We read the CSV files directly from the GitHub repository of the course, so there is nothing to upload.

- **pandas** reads the file into the memory of *this* machine.
- **Spark** DataFrames can be created from a pandas DataFrame with `spark.createDataFrame(...)`. With real big data you would not pass through pandas: Spark would read the files directly from distributed storage, for example with `spark.read.csv(...)`.

In [ ]:
import pandas as pd

DATA_URL = "https://raw.githubusercontent.com/Naviden/Introduction-to-Big-Data-and-NoSQL/main/data"

# pandas
customers_pd = pd.read_csv(f"{DATA_URL}/retail/customers.csv")
transactions_pd = pd.read_csv(f"{DATA_URL}/retail/transactions.csv")

In [ ]:
# Spark (pandas NaN values become Spark nulls)
def to_spark(pdf):
    return spark.createDataFrame(pdf.astype(object).where(pdf.notna(), None))

customers_sp = to_spark(customers_pd)
transactions_sp = to_spark(transactions_pd)

## 2. Looking at the data

| What | pandas | Spark |
|------|--------|-------|
| first rows | `df.head(5)` | `df.show(5)` or `display(df)` |
| number of rows | `len(df)` | `df.count()` |
| columns and types | `df.dtypes` | `df.printSchema()` |

In [ ]:
# pandas
customers_pd.head(5)

In [ ]:
# Spark
customers_sp.show(5)

In Databricks, `display` gives an interactive table: you can sort the columns, and with the **+** button above the table you can create a chart.

In [ ]:
display(customers_sp)

In [ ]:
# pandas: number of rows
len(customers_pd)

In [ ]:
# Spark: number of rows (should be 250)
customers_sp.count()

In [ ]:
# pandas: column types
customers_pd.dtypes

In [ ]:
# Spark: column types
customers_sp.printSchema()

## 3. Selecting columns

In pandas you use square brackets, in Spark the `select` method.

In [ ]:
# pandas
customers_pd[["first_name", "last_name", "age"]].head(5)

In [ ]:
# Spark
customers_sp.select("first_name", "last_name", "age").show(5)

## 4. Filtering rows

Both use conditions on columns. In Spark you can write the column as `customers_sp["age"]` or as `F.col("age")`. When you combine conditions, use `&` (and), `|` (or), and put each condition in parentheses: this is the same in pandas and Spark.

In [ ]:
# pandas: customers older than 30
customers_pd[customers_pd["age"] > 30].head(5)

In [ ]:
# Spark: customers older than 30
customers_sp.filter(F.col("age") > 30).show(5)

In [ ]:
# pandas: Gold customers who are students
customers_pd[(customers_pd["loyalty_tier"] == "Gold") & (customers_pd["is_student"] == 1)].head(5)

In [ ]:
# Spark: Gold customers who are students
customers_sp.filter((F.col("loyalty_tier") == "Gold") & (F.col("is_student") == 1)).show(5)

## 5. Adding a new column

- pandas **changes** the DataFrame in place when you assign a new column.
- Spark DataFrames **never change**: `withColumn` returns a *new* DataFrame, so you have to save it in a variable.

In [ ]:
# pandas
customers_pd["full_name"] = customers_pd["first_name"] + " " + customers_pd["last_name"]
customers_pd[["customer_id", "full_name"]].head(5)

In [ ]:
# Spark
customers_sp = customers_sp.withColumn("full_name", F.concat_ws(" ", "first_name", "last_name"))
customers_sp.select("customer_id", "full_name").show(5)

A column with conditions: an age group for each customer.

In [ ]:
# pandas
customers_pd["age_group"] = pd.cut(customers_pd["age"], bins=[0, 29, 50, 200], labels=["<30", "30-50", ">50"])
customers_pd[["age", "age_group"]].head(5)

In [ ]:
# Spark
customers_sp = customers_sp.withColumn(
    "age_group",
    F.when(F.col("age") < 30, "<30").when(F.col("age") <= 50, "30-50").otherwise(">50")
)
customers_sp.select("age", "age_group").show(5)

## 6. Sorting

In [ ]:
# pandas: the 5 oldest customers
customers_pd.sort_values("age", ascending=False).head(5)

In [ ]:
# Spark: the 5 oldest customers
customers_sp.orderBy(F.desc("age")).show(5)

## 7. Grouping and aggregating

| What | pandas | Spark |
|------|--------|-------|
| count per group | `df["col"].value_counts()` | `df.groupBy("col").count()` |
| several statistics | `df.groupby("col").agg(...)` | `df.groupBy("col").agg(...)` |

In [ ]:
# pandas: customers per loyalty tier
customers_pd["loyalty_tier"].value_counts()

In [ ]:
# Spark: customers per loyalty tier
customers_sp.groupBy("loyalty_tier").count().orderBy(F.desc("count")).show()

In [ ]:
# pandas: number of customers and average age per region
customers_pd.groupby("home_region").agg(
    customers=("customer_id", "count"),
    avg_age=("age", "mean"),
).sort_values("customers", ascending=False).head(5)

In [ ]:
# Spark: number of customers and average age per region
(customers_sp
    .groupBy("home_region")
    .agg(F.count("customer_id").alias("customers"),
         F.round(F.avg("age"), 1).alias("avg_age"))
    .orderBy(F.desc("customers"))
    .show(5))

## 8. Missing values

In the transactions, not every customer left a review, so `review_rating` has missing values. pandas calls them `NaN`, Spark calls them `null`.

In [ ]:
# pandas: missing values per column
transactions_pd.isna().sum()

In [ ]:
# Spark: missing values per column
transactions_sp.select(
    [F.sum(F.col(c).isNull().cast("int")).alias(c) for c in transactions_sp.columns]
).show()

In [ ]:
# pandas: replace missing ratings with 0
transactions_pd.fillna({"review_rating": 0})["review_rating"].isna().sum()

In [ ]:
# Spark: replace missing ratings with 0
transactions_sp.fillna({"review_rating": 0}).filter(F.col("review_rating").isNull()).count()

## 9. Removing duplicates

The transactions file contains a few duplicated rows on purpose.

In [ ]:
# pandas
print("rows:", len(transactions_pd), "| without duplicates:", len(transactions_pd.drop_duplicates()))

In [ ]:
# Spark
print("rows:", transactions_sp.count(), "| without duplicates:", transactions_sp.dropDuplicates().count())

## 10. Joining two tables

pandas uses `merge`, Spark uses `join`. Here we add the customer data to each transaction, then compute the revenue per loyalty tier.

In [ ]:
# pandas
joined_pd = transactions_pd.merge(customers_pd, on="customer_id", how="inner")
joined_pd.groupby("loyalty_tier")["total_amount"].sum().round(2).sort_values(ascending=False)

In [ ]:
# Spark
joined_sp = transactions_sp.join(customers_sp, on="customer_id", how="inner")
(joined_sp
    .groupBy("loyalty_tier")
    .agg(F.round(F.sum("total_amount"), 2).alias("total_amount"))
    .orderBy(F.desc("total_amount"))
    .show())

## 11. The big difference: lazy evaluation

In pandas every line is executed immediately. In Spark, operations like `select`, `filter`, `withColumn`, `groupBy` and `join` are **transformations**: Spark only writes down *what* to do. The work starts only when you call an **action**, like `show()`, `count()` or `collect()`.

Run the next cell: it returns immediately, even if the data were huge, because nothing is computed yet.

In [ ]:
plan = (transactions_sp
    .filter(F.col("total_amount") > 100)
    .select("transaction_id", "store_id", "total_amount")
    .groupBy("store_id")
    .count())

print(type(plan))  # still just a DataFrame: no result yet

Now we ask for a result: this is when Spark actually reads and processes the data.

In [ ]:
plan.orderBy("store_id").show()

With `explain()` you can see the plan Spark built. Before running it, Spark optimizes the whole chain: for example, it applies the filter as early as possible.

In [ ]:
plan.explain()

## 12. From Spark back to pandas

`toPandas()` brings the result to this machine as a pandas DataFrame, for example to make a chart. Do it only with **small** results: the whole DataFrame has to fit in the memory of one machine.

In [ ]:
summary_pd = joined_sp.groupBy("loyalty_tier").count().toPandas()
summary_pd

## Summary

| What | pandas | Spark |
|------|--------|-------|
| first rows | `df.head()` | `df.show()`, `display(df)` |
| number of rows | `len(df)` | `df.count()` |
| columns and types | `df.dtypes` | `df.printSchema()` |
| select columns | `df[["a", "b"]]` | `df.select("a", "b")` |
| filter rows | `df[df["a"] > 1]` | `df.filter(F.col("a") > 1)` |
| new column | `df["c"] = ...` | `df = df.withColumn("c", ...)` |
| sort | `df.sort_values("a")` | `df.orderBy("a")` |
| group and aggregate | `df.groupby("a").agg(...)` | `df.groupBy("a").agg(...)` |
| missing values | `df.isna()`, `df.fillna()` | `F.col("a").isNull()`, `df.fillna()` |
| duplicates | `df.drop_duplicates()` | `df.dropDuplicates()` |
| join | `df1.merge(df2, on="k")` | `df1.join(df2, on="k")` |
| execution | immediate | lazy: only when an action is called |
| where the data is | memory of one machine | split across the machines of the cluster |

## Exercises

Solve each exercise with **Spark** (and with pandas too, if you want to compare).

1. How many transactions were paid with each payment method (`payment_method`)?
2. What is the average `total_amount` per sales channel (`channel`)? Round it to 2 decimals.
3. Find the 5 customers who spent the most in total. Show their full name and the total amount.
4. How many customers never left a review (all their `review_rating` values are missing)?

In [ ]:
# Exercise 1

In [ ]:
# Exercise 2

In [ ]:
# Exercise 3

In [ ]:
# Exercise 4